In [13]:
import torch
from torch import nn
import math

In [14]:
class SimpleEncoderRegressor(nn.Module):
    def __init__(self, vocab_size, out_shape, d_model=64, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.out_shape = out_shape
        self.K, self.Q, self.P = out_shape
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.proj = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128), 
            nn.GELU(),
            nn.Linear(128, self.K * self.Q * self.P),
        )
        

    def forward(self, tok_ids):  # tok_ids: [B, L]
        x = self.emb(tok_ids)                 # [B, L, C]
        mask = (tok_ids != self.pad_id).float().unsqueeze(-1)  # [B, L, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp_min(1.0)                 # [B,1,1] -> [B,1,1]
        pooled = x.sum(dim=1) / denom.squeeze(1)               # [B, C]
        out = self.proj(pooled)                                 # [B, S*Q*A]
        return out.view(-1, self.K, self.Q, self.P)
    
simple_model_1 = SimpleEncoderRegressor(vocab_size=20, out_shape=(2, 3, 3))
print(simple_model_1)

SimpleEncoderRegressor(
  (emb): Embedding(20, 64, padding_idx=0)
  (proj): Sequential(
    (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=64, out_features=128, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=128, out_features=18, bias=True)
  )
)


In [16]:
# ---------------- Positional encoding (batch_first) ----------------
class PositionalEncodingBF(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)                                  # [L, C]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # [L, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, L, C] for batch_first broadcasting
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):  # x: [B, L, C]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# ---------------- Simple Transformer Encoder -> angle regressor ----------------
class SimpleTransformer(nn.Module):
    def __init__(self,
                 vocab_size: int,
                 out_shape: tuple,   # (K, Q, P)
                 d_model: int = 128,
                 nhead: int = 2,
                 num_encoder_layers: int = 2,
                 dim_feedforward: int = 256,
                 dropout: float = 0.1,
                 pad_id: int = 0,
                 max_len: int = 128):
        super().__init__()
        self.K, self.Q, self.P = out_shape
        self.out_dim = self.K * self.Q * self.P
        self.d_model = d_model
        self.pad_id = pad_id

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = PositionalEncodingBF(d_model, dropout, max_len=max_len)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)

        self.regression_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, self.out_dim)
        )

    def forward(self, src_tokens):  # [B, L]
        # Embedding + scale
        x = self.embedding(src_tokens) * math.sqrt(self.d_model)  # [B, L, C]
        x = self.pos_encoder(x)

        # Key padding mask: True where padding should be ignored
        key_padding_mask = (src_tokens == self.pad_id)  # [B, L]

        # Encoder
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)  # [B, L, C]

        # Masked mean pool over tokens
        mask = (~key_padding_mask).unsqueeze(-1).float()  # [B, L, 1]
        denom = mask.sum(dim=1).clamp_min(1.0)            # [B, 1, 1]
        pooled = (x * mask).sum(dim=1) / denom.squeeze(1) # [B, C]

        # Regress angles
        y = self.regression_head(pooled)                  # [B, K*Q*P]
        return y.view(-1, self.K, self.Q, self.P)         # [B, K, Q, P]

simple_model_2 = SimpleTransformer(vocab_size=12, out_shape=(2, 3, 3))
print(simple_model_2)

SimpleTransformer(
  (embedding): Embedding(12, 128, padding_idx=0)
  (pos_encoder): PositionalEncodingBF(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (regression_head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_featur